# Prototype & Live Demo Klasifikasi Kesegaran Daun Kampus (Progress 3)
**Mata Kuliah:** Pengolahan Citra Digital dan Visi Komputer (CIF61339), Kelas A  
**Dosen Pengampu:** Yuita Arum Sari, S.Kom., M.Kom., Ph.D.  
**Fakultas Ilmu Komputer (FILKOM), Universitas Brawijaya**

**Tim Topik 2:**
1. **Nadhif Rif'at Rasendriya (235150201111074):** Data, Audit, Pipeline PCD v2, Validasi E7, & Integrasi Demo
2. **Nadhif Adita Fernanda (235150201111033):** Ekstraksi 28 Fitur & Classical Machine Learning
3. **Muzhaffar Ammar (235150200111033):** Deep Learning (MobileNetV2, Grad-CAM) & Object Detection

***
### Panduan Singkat Penggunaan (Dry-Run & Presentasi Rabu)
Notebook ini menggabungkan seluruh pipeline **PCD v2 (Gamma Adaptif L* = 0,716, Bilateral Filter, CLAHE 2.0, Segmentasi Otsu b*+S + Morfologi + LCC, ROI 224x224)** dengan tiga keluarga model secara berdampingan:
1. **Classical ML (`final_best_ml_model.pkl`):** 28 fitur warna (HSV/CIE-LAB) dan tekstur (GLCM bebas latar + LBP).
2. **Deep Learning Klasifikasi (`best_mobilenetv2_C2_*.keras`):** MobileNetV2 pada citra C2 beserta peta atensi **Grad-CAM**.
3. **Deep Learning Deteksi Objek (`best_yolov8n_c2_nohsv.pt`):** YOLOv8n terlatih pada citra C2 tanpa distorsi warna HSV.

**Tiga Mode Demo yang Tersedia:**
* **Mode 1 (Uji Batch Sesi 3):** Simpan foto daun baru ke folder Google Drive `Developments/Uji_Sesi3_Rabu` (taruh di folder utama atau pisah ke subfolder `Sehat/`, `Menguning/`, `Kering/` agar akurasi eksternal terhitung otomatis). Jika folder masih kosong, mode *dry-run* otomatis mengetes 3 sampel asli dari dataset.
* **Mode 2 (Live Upload Foto):** Unggah langsung 1 foto baru dari HP/laptop saat presentasi di kelas.
* **Mode 3 (Live Webcam Colab):** Ambil gambar langsung dari kamera laptop di depan dosen (letakkan daun di atas kertas putih polos).

## 1. Setup Lingkungan, Mount Google Drive, & Pemuatan Otomatis Model
Jalankan sel kode di bawah ini terlebih dahulu untuk menghubungkan Google Drive, mendeteksi seluruh model terlatih yang tersedia pada `Dataset_P2_Processed_v2`, dan melakukan *warm-up* inferensi agar pengukuran latensi (ms/citra) akurat.

In [ ]:
# === BAGIAN 1: SETUP, MOUNT DRIVE, & LOAD 3 MODEL (ML + MOBILENETV2 + YOLOV8N) ===
import os
import re
import glob
import time
import warnings
warnings.filterwarnings('ignore')

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from google.colab import drive, files

try:
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix, greycoprops as graycoprops, local_binary_pattern

try:
    from ultralytics import YOLO
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
    from ultralytics import YOLO

import tensorflow as tf
from tensorflow.keras import models as keras_models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess

drive.mount('/content/drive')

# Direktori Utama Proyek di Google Drive
DEV_DIR         = '/content/drive/MyDrive/Project Akhir Computer Vision/Developments'
RAW_DATASET_DIR = os.path.join(DEV_DIR, 'Dataset')
OUT_V2          = os.path.join(DEV_DIR, 'Dataset_P2_Processed_v2')
OUT_V1          = os.path.join(DEV_DIR, 'Dataset_P2_Processed')
TEST_SESI3_DIR  = os.path.join(DEV_DIR, 'Uji_Sesi3_Rabu')
DEMO_OUT_DIR    = os.path.join(DEV_DIR, 'Progress_3_Outputs', 'Demo_Hasil')

os.makedirs(TEST_SESI3_DIR, exist_ok=True)
os.makedirs(DEMO_OUT_DIR, exist_ok=True)

CLASS_NAMES  = ['Sehat', 'Menguning', 'Kering']
IDX_TO_LABEL = {i: name for i, name in enumerate(CLASS_NAMES)}
COLOR_MAP    = {'Sehat': '#16a34a', 'Menguning': '#eab308', 'Kering': '#b45309'}

# Cek status GPU
gpus = tf.config.list_physical_devices('GPU')
DEVICE_INFO = gpus[0].name if gpus else 'CPU'
print(f"Perangkat komputasi aktif : {DEVICE_INFO}")

# ==============================================================================
# DETEKSI & MUAT OTOMATIS 3 KELUARGA MODEL
# ==============================================================================
LOADED_MODELS = {'ml': None, 'mobilenetv2': None, 'yolov8n': None}
MODEL_META = {}

# 1. Classical ML (.pkl dari Notebook 03 Nadhif Adita)
ml_v2_path = os.path.join(OUT_V2, 'models', 'final_best_ml_model.pkl')
ml_v1_path = os.path.join(OUT_V1, 'models', 'final_best_ml_model.pkl')

if os.path.exists(ml_v2_path):
    LOADED_MODELS['ml'] = joblib.load(ml_v2_path)
    MODEL_META['ml'] = f"ML Klasik v2 ({LOADED_MODELS['ml'].get('model_name', 'Best ML')})"
    print(f"[OK]   {MODEL_META['ml']} dimuat dari: {ml_v2_path}")
elif os.path.exists(ml_v1_path):
    LOADED_MODELS['ml'] = joblib.load(ml_v1_path)
    MODEL_META['ml'] = f"ML Klasik FALLBACK v1 ({LOADED_MODELS['ml'].get('model_name', 'Best ML')})"
    print(f"[INFO] Model ML v2 belum ditemukan, memakai fallback v1 dari: {ml_v1_path}")
    print("       Pastikan Notebook 03 dijalankan ulang pada Dataset_P2_Processed_v2 sebelum Rabu.")
else:
    print("[SKIP] Model ML Klasik (.pkl) belum ditemukan di Drive.")

# 2. Deep Learning Klasifikasi: MobileNetV2 C2 (.keras dari Notebook 04 Ammar)
mnv2_candidates = sorted(glob.glob(os.path.join(OUT_V2, 'Models_DL_MobileNetV2', 'best_mobilenetv2_C2*.keras')))
if mnv2_candidates:
    mnv2_path = mnv2_candidates[-1]
    LOADED_MODELS['mobilenetv2'] = keras_models.load_model(mnv2_path)
    MODEL_META['mobilenetv2'] = f"MobileNetV2 C2 ({os.path.basename(mnv2_path)})"
    # Warm-up 1 kali agar pengukuran latensi inferensi pertama akurat
    _ = LOADED_MODELS['mobilenetv2'].predict(np.zeros((1, 224, 224, 3), dtype=np.float32), verbose=0)
    print(f"[OK]   {MODEL_META['mobilenetv2']} dimuat dari: {mnv2_path}")
else:
    print("[SKIP] Model MobileNetV2 C2 (.keras) belum ditemukan di Models_DL_MobileNetV2.")

# 3. Deep Learning Deteksi Objek: YOLOv8n C2 No-HSV (.pt)
yolo_candidates = [
    os.path.join(OUT_V2, 'Models_DL_MobileNetV2', 'best_yolov8n_c2_nohsv.pt'),
    os.path.join(OUT_V2, 'runs_yolo', 'leaf_yolov8n_c2_nohsv', 'weights', 'best.pt'),
    os.path.join(OUT_V2, 'Models_DL_MobileNetV2', 'best_yolov8n.pt')
]
yolo_path = next((p for p in yolo_candidates if os.path.exists(p)), None)
if yolo_path:
    LOADED_MODELS['yolov8n'] = YOLO(yolo_path)
    MODEL_META['yolov8n'] = f"YOLOv8n ({os.path.basename(yolo_path)})"
    # Warm-up 1 kali
    _ = LOADED_MODELS['yolov8n'].predict(np.zeros((224, 224, 3), dtype=np.uint8), imgsz=224, verbose=False)
    print(f"[OK]   {MODEL_META['yolov8n']} dimuat dari: {yolo_path}")
else:
    print("[SKIP] Model YOLOv8n (.pt) belum ditemukan di Drive.")

active_count = sum(v is not None for v in LOADED_MODELS.values())
print(f"\nRingkasan: {active_count} / 3 model siap digunakan untuk demo.")
print(f"Folder unggah foto Sesi 3 : {TEST_SESI3_DIR}")
print(f"Folder output hasil demo  : {DEMO_OUT_DIR}")

## 2. Pipeline PCD v2, Ekstraksi 28 Fitur, Mesin Inferensi Multi-Model, & Dashboard Visualisasi
Seluruh fungsi pada bagian ini sinkron 100% dengan Notebook 02, 03, dan 04:
* **PCD v2:** Resize proporsional sisi panjang 800 px, Gamma Adaptif `L**gamma` menuju target median `L* = 0,716` (`[0,8; 1,25]`), Bilateral Filter (`d=9, sigma=75`), CLAHE pada kanal `L*` (`clipLimit=2,0`), Dual Otsu (`b* OR S`) + Morfologi Open/Close (`7x7`) + Largest Connected Component, serta *letterbox ROI crop* 224x224 dengan padding 8%.
* **Ekstraksi 28 Fitur:** 12 fitur statistik warna HSV dan CIE-LAB khusus piksel daun, 6 fitur GLCM bebas latar (intensitas daun digeser ke `[1..255]`), serta 10 bin histogram Uniform LBP (`P=8, R=1`).
* **Inferensi & Dashboard 5 Panel:** Menampilkan citra mentah, mask Otsu, C2 ROI beserta kotak deteksi YOLOv8n, peta atensi Grad-CAM MobileNetV2, dan grafik perbandingan *confidence*, latensi (ms), serta **Konsensus Multi-Model**.

In [ ]:
# === BAGIAN 2: PIPELINE PCD V2, EKSTRAKSI 28 FITUR, INFERENSI, & DASHBOARD ===
WORK_LONG = 800
TARGET_L  = 0.716

COLOR_COLS = [
    'H_mean', 'H_std', 'S_mean', 'S_std', 'V_mean', 'V_std',
    'L_mean', 'L_std', 'a_mean', 'a_std', 'b_mean', 'b_std'
]
GLCM_PROPS = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
GLCM_COLS  = [f'GLCM_{p}' for p in GLCM_PROPS]
LBP_COLS   = [f'LBP_bin_{i}' for i in range(10)]
ALL_FEATURE_COLS = COLOR_COLS + GLCM_COLS + LBP_COLS

# === A. FUNGSI PIPELINE PCD V2 ===
def resize_long(img, long_side=WORK_LONG):
    h, w = img.shape[:2]
    s = long_side / max(h, w)
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)

def adaptive_gamma_lab(img_bgr, target=TARGET_L, lo=0.8, hi=1.25):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    m = float(np.clip(lab[:, :, 0].mean() / 255.0, 0.05, 0.95))
    g = float(np.clip(np.log(target) / np.log(m), lo, hi))
    lut = (((np.arange(256) / 255.0) ** g) * 255).astype(np.uint8)
    lab[:, :, 0] = cv2.LUT(lab[:, :, 0], lut)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), g

def apply_bilateral(img_bgr, d=9, sigma_color=75, sigma_space=75):
    return cv2.bilateralFilter(img_bgr, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)

def apply_clahe_lab(img_bgr, clip_limit=2.0, tile_size=(8, 8)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    l_clahe = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_LAB2BGR)

def segment_leaf_otsu_morph(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, _, b = cv2.split(lab)
    _, s, _ = cv2.split(hsv)
    _, mask_b = cv2.threshold(b, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask_s = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask_raw = cv2.bitwise_or(mask_b, mask_s)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_clean = cv2.morphologyEx(mask_raw, cv2.MORPH_OPEN, kernel, iterations=1)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel, iterations=2)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_clean, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask_clean = np.where(labels == largest_label, 255, 0).astype(np.uint8)
    return mask_raw, mask_clean

def crop_roi_with_padding(img_bgr, mask, target_size=(224, 224), pad_ratio=0.08):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return cv2.resize(img_bgr, target_size), cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    x1, x2 = np.min(xs), np.max(xs)
    y1, y2 = np.min(ys), np.max(ys)
    h_box, w_box = y2 - y1, x2 - x1
    max_side = int(max(h_box, w_box) * (1.0 + 2 * pad_ratio))
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    half = max_side // 2
    img_masked = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    h_img, w_img = img_bgr.shape[:2]
    canvas_img = np.zeros((max_side, max_side, 3), dtype=np.uint8)
    canvas_mask = np.zeros((max_side, max_side), dtype=np.uint8)
    src_x1, src_y1 = max(0, cx - half), max(0, cy - half)
    src_x2, src_y2 = min(w_img, cx - half + max_side), min(h_img, cy - half + max_side)
    dst_x1, dst_y1 = src_x1 - (cx - half), src_y1 - (cy - half)
    dst_x2, dst_y2 = dst_x1 + (src_x2 - src_x1), dst_y1 + (src_y2 - src_y1)
    canvas_img[dst_y1:dst_y2, dst_x1:dst_x2] = img_masked[src_y1:src_y2, src_x1:src_x2]
    canvas_mask[dst_y1:dst_y2, dst_x1:dst_x2] = mask[src_y1:src_y2, src_x1:src_x2]
    roi_resized = cv2.resize(canvas_img, target_size, interpolation=cv2.INTER_AREA)
    mask_resized = cv2.resize(canvas_mask, target_size, interpolation=cv2.INTER_NEAREST)
    roi_resized = cv2.bitwise_and(roi_resized, roi_resized, mask=mask_resized)
    return roi_resized, mask_resized

def run_pcd_v2(img_bgr):
    t0 = time.perf_counter()
    im_work = resize_long(img_bgr, WORK_LONG)
    im_gamma, gamma_val = adaptive_gamma_lab(im_work, TARGET_L)
    im_c1 = apply_clahe_lab(apply_bilateral(im_gamma), clip_limit=2.0)
    _, mask_full = segment_leaf_otsu_morph(im_c1)
    c2_roi, mask_roi = crop_roi_with_padding(im_c1, mask_full, target_size=(224, 224))
    pcd_ms = (time.perf_counter() - t0) * 1000.0
    leaf_area_pct = float((mask_full > 0).mean() * 100.0)
    return {
        'im_work': im_work,
        'im_c1': im_c1,
        'mask_full': mask_full,
        'c2_roi': c2_roi,
        'mask_roi': mask_roi,
        'gamma_val': gamma_val,
        'leaf_area_pct': leaf_area_pct,
        'pcd_ms': pcd_ms
    }

# === B. EKSTRAKSI 28 FITUR HANDCRAFTED (SINKRON NOTEBOOK 03) ===
def extract_handcrafted_features(img_bgr, mask=None, use_mask=True):
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    valid = (mask > 0) if (use_mask and mask is not None and np.sum(mask > 0) > 50) else np.ones(gray.shape, dtype=bool)

    feats = {}
    for idx, ch_name in enumerate(['H', 'S', 'V']):
        vals = hsv[:, :, idx][valid]
        feats[f'{ch_name}_mean'] = float(np.mean(vals))
        feats[f'{ch_name}_std']  = float(np.std(vals))
    for idx, ch_name in enumerate(['L', 'a', 'b']):
        vals = lab[:, :, idx][valid]
        feats[f'{ch_name}_mean'] = float(np.mean(vals))
        feats[f'{ch_name}_std']  = float(np.std(vals))

    if use_mask and mask is not None and np.sum(mask > 0) > 50:
        gray_glcm = np.where(valid, np.clip(gray, 1, 255), 0).astype(np.uint8)
        glcm = graycomatrix(
            gray_glcm, distances=[1, 2],
            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
            levels=256, symmetric=True, normed=False
        )
        glcm[0, :, :, :] = 0
        glcm[:, 0, :, :] = 0
        glcm_sum = np.sum(glcm, axis=(0, 1), keepdims=True)
        glcm_norm = np.divide(glcm.astype(np.float64), glcm_sum, out=np.zeros_like(glcm, dtype=np.float64), where=glcm_sum > 0)
    else:
        glcm_norm = graycomatrix(
            gray, distances=[1, 2],
            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
            levels=256, symmetric=True, normed=True
        )
    for prop in GLCM_PROPS:
        feats[f'GLCM_{prop}'] = float(np.mean(graycoprops(glcm_norm, prop)))

    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp[valid], bins=np.arange(0, 11), density=True)
    for i in range(10):
        feats[f'LBP_bin_{i}'] = float(hist[i])
    return feats

# === C. GRAD-CAM MOBILENETV2 ===
def compute_gradcam_overlay(c2_bgr, model, last_conv_layer_name='out_relu', alpha=0.45):
    img_rgb = cv2.cvtColor(c2_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    img_tensor = np.expand_dims(mnv2_preprocess(img_rgb), axis=0)
    grad_model = keras_models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_outputs[0] @ pooled_grads[..., tf.newaxis]).numpy()
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0:
        heatmap = heatmap / np.max(heatmap)
    heatmap_resized = cv2.resize(heatmap, (c2_bgr.shape[1], c2_bgr.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(c2_bgr, 1 - alpha, heatmap_color, alpha, 0)
    return cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)

# === D. MESIN INFERENSI MULTI-MODEL ===
def run_multimodel_inference(img_bgr, filename='sample.jpg', true_label=None):
    pcd = run_pcd_v2(img_bgr)
    c2_roi   = pcd['c2_roi']
    mask_roi = pcd['mask_roi']
    preds = {}

    # 1. Prediksi Classical ML (28 fitur)
    if LOADED_MODELS['ml'] is not None:
        t0 = time.perf_counter()
        bundle = LOADED_MODELS['ml']
        feats = extract_handcrafted_features(c2_roi, mask=mask_roi, use_mask=True)
        feat_cols = bundle.get('feature_cols', ALL_FEATURE_COLS)
        x_in = np.array([[feats[c] for c in feat_cols]], dtype=np.float64)
        pipe = bundle['pipeline']
        if hasattr(pipe, 'predict_proba'):
            probs_raw = pipe.predict_proba(x_in)[0]
            cls_order = list(pipe.classes_)
            prob_dict = {c: float(probs_raw[cls_order.index(c)]) for c in CLASS_NAMES if c in cls_order}
            pred_lbl = max(prob_dict, key=prob_dict.get)
            conf_ml = prob_dict[pred_lbl] * 100.0
        else:
            pred_lbl = str(pipe.predict(x_in)[0])
            prob_dict = {pred_lbl: 1.0}
            conf_ml = 100.0
        ml_ms = (time.perf_counter() - t0) * 1000.0
        preds['ML_Klasik'] = {
            'label': pred_lbl,
            'conf': conf_ml,
            'probs': prob_dict,
            'ms': ml_ms
        }

    # 2. Prediksi MobileNetV2 C2 + Grad-CAM
    gradcam_rgb = None
    if LOADED_MODELS['mobilenetv2'] is not None:
        t0 = time.perf_counter()
        img_rgb = cv2.cvtColor(c2_roi, cv2.COLOR_BGR2RGB).astype(np.float32)
        x_dl = np.expand_dims(mnv2_preprocess(img_rgb), axis=0)
        probs_dl = LOADED_MODELS['mobilenetv2'].predict(x_dl, verbose=0)[0]
        mnv2_ms = (time.perf_counter() - t0) * 1000.0
        prob_dict = {CLASS_NAMES[i]: float(probs_dl[i]) for i in range(len(CLASS_NAMES))}
        pred_lbl = CLASS_NAMES[int(np.argmax(probs_dl))]
        try:
            gradcam_rgb = compute_gradcam_overlay(c2_roi, LOADED_MODELS['mobilenetv2'])
        except Exception:
            gradcam_rgb = None
        preds['MobileNetV2'] = {
            'label': pred_lbl,
            'conf': prob_dict[pred_lbl] * 100.0,
            'probs': prob_dict,
            'ms': mnv2_ms
        }

    # 3. Prediksi YOLOv8n C2 No-HSV
    yolo_bbox = None
    if LOADED_MODELS['yolov8n'] is not None:
        t0 = time.perf_counter()
        y_out = LOADED_MODELS['yolov8n'].predict(c2_roi, imgsz=224, conf=0.15, verbose=False)[0]
        yolo_ms = (time.perf_counter() - t0) * 1000.0
        if len(y_out.boxes) > 0:
            best_i = int(y_out.boxes.conf.argmax().item())
            cls_id = int(y_out.boxes.cls[best_i].item())
            conf_y = float(y_out.boxes.conf[best_i].item()) * 100.0
            yolo_bbox = y_out.boxes.xyxy[best_i].cpu().numpy().astype(int).tolist()
            pred_lbl = IDX_TO_LABEL.get(cls_id, 'Unknown')
        else:
            pred_lbl = 'Tidak Terdeteksi'
            conf_y = 0.0
        preds['YOLOv8n'] = {
            'label': pred_lbl,
            'conf': conf_y,
            'bbox': yolo_bbox,
            'ms': yolo_ms
        }

    # 4. Konsensus Multi-Model (Majority Vote, tie-breaker rata-rata confidence tertinggi)
    valid_votes = [(v['label'], v['conf']) for v in preds.values() if v['label'] in CLASS_NAMES]
    if valid_votes:
        counts = Counter([lbl for lbl, _ in valid_votes])
        top_count = counts.most_common(1)[0][1]
        top_labels = [lbl for lbl, cnt in counts.items() if cnt == top_count]
        if len(top_labels) == 1:
            consensus_lbl = top_labels[0]
        else:
            consensus_lbl = max(top_labels, key=lambda l: np.mean([c for lbl, c in valid_votes if lbl == l]))
        agree_str = f"{counts[consensus_lbl]}/{len(valid_votes)} model sepakat"
    else:
        consensus_lbl = 'N/A'
        agree_str = 'Tidak ada model aktif'

    return {
        'filename': filename,
        'true_label': true_label,
        'pcd': pcd,
        'preds': preds,
        'gradcam_rgb': gradcam_rgb,
        'yolo_bbox': yolo_bbox,
        'consensus': consensus_lbl,
        'agreement': agree_str
    }

# === E. VISUALISASI DASHBOARD EKSEKUTIF 5 PANEL ===
def visualize_demo_result(res, save_path=None):
    pcd = res['pcd']
    preds = res['preds']

    fig, axes = plt.subplots(1, 5, figsize=(21, 4.4))

    # Panel 1: Foto Mentah Asli
    axes[0].imshow(cv2.cvtColor(pcd['im_work'], cv2.COLOR_BGR2RGB))
    ttl_1 = f"1. Input Mentah\n{res['filename']}"
    if res['true_label']:
        ttl_1 += f"\n[Aktual: {res['true_label']}]"
    axes[0].set_title(ttl_1, fontsize=10, fontweight='bold')
    axes[0].axis('off')

    # Panel 2: Mask Otsu b*+S + Morfologi
    axes[1].imshow(pcd['mask_full'], cmap='gray')
    axes[1].set_title(
        f"2. Otsu b*+S + LCC\nGamma={pcd['gamma_val']:.2f} | Area={pcd['leaf_area_pct']:.1f}%\nPCD: {pcd['pcd_ms']:.1f} ms",
        fontsize=10, fontweight='bold'
    )
    axes[1].axis('off')

    # Panel 3: C2 ROI 224x224 + Bounding Box YOLOv8n
    c2_disp = cv2.cvtColor(pcd['c2_roi'], cv2.COLOR_BGR2RGB).copy()
    if res['yolo_bbox'] is not None:
        x1, y1, x2, y2 = res['yolo_bbox']
        cv2.rectangle(c2_disp, (x1, y1), (x2, y2), (34, 197, 94), 2)
        y_info = preds['YOLOv8n']
        ttl_3 = f"3. C2 ROI + YOLOv8n\nPred: {y_info['label']} ({y_info['conf']:.1f}%)\nInfer: {y_info['ms']:.1f} ms"
    else:
        ttl_3 = "3. C2 ROI (224x224)\n(YOLOv8n nonaktif / 0 box)"
    axes[2].imshow(c2_disp)
    axes[2].set_title(ttl_3, fontsize=10, fontweight='bold')
    axes[2].axis('off')

    # Panel 4: Grad-CAM MobileNetV2
    if res['gradcam_rgb'] is not None:
        m_info = preds['MobileNetV2']
        axes[3].imshow(res['gradcam_rgb'])
        axes[3].set_title(
            f"4. Grad-CAM MobileNetV2\nPred: {m_info['label']} ({m_info['conf']:.1f}%)\nInfer: {m_info['ms']:.1f} ms",
            fontsize=10, fontweight='bold'
        )
    else:
        axes[3].imshow(cv2.cvtColor(pcd['c2_roi'], cv2.COLOR_BGR2RGB))
        axes[3].set_title("4. Grad-CAM MobileNetV2\n(Model .keras belum dimuat)", fontsize=10)
    axes[3].axis('off')

    # Panel 5: Ringkasan Confidence & Konsensus Multi-Model
    ax5 = axes[4]
    model_labels, conf_vals, bar_colors, ann_texts = [], [], [], []
    for m_key, nice_name in [('ML_Klasik', 'ML Klasik'), ('MobileNetV2', 'MobileNetV2'), ('YOLOv8n', 'YOLOv8n')]:
        if m_key in preds:
            p_lbl = preds[m_key]['label']
            p_cnf = preds[m_key]['conf']
            p_ms  = preds[m_key]['ms']
            model_labels.append(nice_name)
            conf_vals.append(p_cnf)
            bar_colors.append(COLOR_MAP.get(p_lbl, '#64748b'))
            ann_texts.append(f"{p_lbl} ({p_cnf:.1f}%, {p_ms:.1f} ms)")

    y_pos = np.arange(len(model_labels))
    bars = ax5.barh(y_pos, conf_vals, color=bar_colors, edgecolor='black', height=0.52)
    ax5.set_yticks(y_pos)
    ax5.set_yticklabels(model_labels, fontsize=9.5, fontweight='bold')
    ax5.set_xlim(0, 120)
    ax5.set_xlabel('Confidence (%)', fontsize=9)
    ax5.invert_yaxis()
    ax5.grid(axis='x', alpha=0.3)
    for bar, txt in zip(bars, ann_texts):
        w = bar.get_width()
        x_txt = 4 if w >= 55 else w + 2
        txt_col = 'white' if w >= 55 else 'black'
        ax5.text(x_txt, bar.get_y() + bar.get_height()/2, txt,
                 va='center', ha='left', fontsize=8.5, fontweight='bold', color=txt_col)

    cons_color = '#16a34a' if (res['true_label'] is None or res['consensus'] == res['true_label']) else '#dc2626'
    ax5.set_title(f"5. Konsensus: {res['consensus']}\n({res['agreement']})",
                  fontsize=10.5, fontweight='bold', color=cons_color)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=220, bbox_inches='tight')
    plt.show()

print("Pipeline PCD v2, Ekstraksi 28 Fitur, & Mesin Inferensi Multi-Model siap dijalankan!")

## 3. Mode 1: Uji Batch Folder Foto Daun Baru (Data Eksternal Sesi 3, Rabu Pagi)
**Cara Penggunaan Rabu Pagi Sebelum Kelas:**
1. Foto 9 hingga 15 helai daun kampus baru di atas kertas putih polos.
2. Unggah foto tersebut ke Google Drive pada folder `Project Akhir Computer Vision/Developments/Uji_Sesi3_Rabu/`.
   * **Opsi A (Dengan Label Aktual):** Buat subfolder `Sehat/`, `Menguning/`, dan `Kering/` lalu masukkan foto sesuai kondisinya agar skrip otomatis menghitung **Akurasi dan Macro-F1 Sesi 3** untuk setiap model.
   * **Opsi B (Tanpa Subfolder):** Taruh foto langsung di folder utama `Uji_Sesi3_Rabu/` untuk prediksi murni.
3. Klik **Run** pada sel di bawah ini. Jika folder masih kosong (saat *dry-run*), skrip otomatis menguji 3 sampel asli dari dataset primer.

In [ ]:
# === BAGIAN 3 (MODE 1): UJI BATCH FOLDER SESI 3 (DRIVE) + AUTO DRY-RUN ===
from sklearn.metrics import accuracy_score, f1_score

DRY_RUN_IF_EMPTY = True   # Otomatis uji 3 sampel dataset jika Uji_Sesi3_Rabu masih kosong
MAX_PLOTS_SHOWN  = 6      # Maksimal jumlah dashboard 5 panel yang ditampilkan ke layar

def collect_test_images(root_dir):
    items = []
    has_subfolders = any(os.path.isdir(os.path.join(root_dir, c)) for c in CLASS_NAMES)
    if has_subfolders:
        for c in CLASS_NAMES:
            c_dir = os.path.join(root_dir, c)
            if not os.path.isdir(c_dir):
                continue
            for fp in sorted(glob.glob(os.path.join(c_dir, '*'))):
                if fp.lower().endswith(('.jpg', '.jpeg', '.png')):
                    items.append((fp, c))
    for fp in sorted(glob.glob(os.path.join(root_dir, '*'))):
        if os.path.isfile(fp) and fp.lower().endswith(('.jpg', '.jpeg', '.png')):
            items.append((fp, None))
    return items

test_items = collect_test_images(TEST_SESI3_DIR)
is_dry_run = False

if len(test_items) == 0 and DRY_RUN_IF_EMPTY:
    is_dry_run = True
    print(f"[INFO DRY-RUN] Folder '{TEST_SESI3_DIR}' masih kosong.")
    print("Mengambil 3 contoh foto (1 Sehat, 1 Menguning, 1 Kering) dari folder Dataset untuk verifikasi alur demo...\n")
    for folder_name, lbl in [('P1_Sehat', 'Sehat'), ('P1_Menguning', 'Menguning'), ('P1_Kering', 'Kering')]:
        f_list = sorted([
            p for p in glob.glob(os.path.join(RAW_DATASET_DIR, folder_name, '*'))
            if p.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        if f_list:
            test_items.append((f_list[-1], lbl))

if len(test_items) == 0:
    print("Tidak ada foto ditemukan. Silakan unggah foto daun baru ke folder Uji_Sesi3_Rabu.")
else:
    mode_tag = "DRY-RUN" if is_dry_run else "UJI EKSTERNAL SESI 3"
    print(f"=== MEMULAI {mode_tag} ({len(test_items)} FOTO) ===")
    summary_rows = []
    for i, (fpath, true_lbl) in enumerate(test_items):
        fname = os.path.basename(fpath)
        img_bgr = cv2.imread(fpath)
        if img_bgr is None:
            print(f"[PERINGATAN] Gagal membaca gambar: {fname}")
            continue

        res = run_multimodel_inference(img_bgr, filename=fname, true_label=true_lbl)
        p = res['preds']
        row = {
            'Filename': fname,
            'Aktual': true_lbl if true_lbl else '-',
            'PCD_ms': round(res['pcd']['pcd_ms'], 1),
            'Gamma': round(res['pcd']['gamma_val'], 2)
        }
        for m_key in ['ML_Klasik', 'MobileNetV2', 'YOLOv8n']:
            if m_key in p:
                row[f'{m_key}_Pred'] = p[m_key]['label']
                row[f'{m_key}_Conf'] = round(p[m_key]['conf'], 1)
                row[f'{m_key}_ms']   = round(p[m_key]['ms'], 1)
        row['Konsensus']   = res['consensus']
        row['Kesepakatan'] = res['agreement']
        summary_rows.append(row)

        if i < MAX_PLOTS_SHOWN:
            save_fig = os.path.join(DEMO_OUT_DIR, f"Demo_{i+1:02d}_{os.path.splitext(fname)[0]}.png")
            visualize_demo_result(res, save_path=save_fig)

    df_demo = pd.DataFrame(summary_rows)
    csv_out = os.path.join(DEMO_OUT_DIR, 'Hasil_Demo_DryRun.csv' if is_dry_run else 'Hasil_Uji_Eksternal_Sesi3.csv')
    df_demo.to_csv(csv_out, index=False)

    print(f"\n=== TABEL RINGKASAN PREDIKSI ({mode_tag}) ===")
    print(df_demo.to_string(index=False))

    labeled_df = df_demo[df_demo['Aktual'].isin(CLASS_NAMES)]
    if len(labeled_df) > 0:
        print(f"\n=== REKAPITULASI AKURASI & LATENSI RATA-RATA (N = {len(labeled_df)} BERLABEL) ===")
        for col_pred, col_ms, nice in [
            ('ML_Klasik_Pred', 'ML_Klasik_ms', 'Classical ML (28 Fitur)'),
            ('MobileNetV2_Pred', 'MobileNetV2_ms', 'MobileNetV2 (C2 Masked)'),
            ('YOLOv8n_Pred', 'YOLOv8n_ms', 'YOLOv8n (C2 No-HSV)'),
            ('Konsensus', None, 'Konsensus Multi-Model')
        ]:
            if col_pred not in labeled_df.columns:
                continue
            acc = accuracy_score(labeled_df['Aktual'], labeled_df[col_pred]) * 100.0
            f1  = f1_score(labeled_df['Aktual'], labeled_df[col_pred], average='macro', zero_division=0) * 100.0
            ms_txt = f" | Avg Infer: {labeled_df[col_ms].mean():.2f} ms" if (col_ms and col_ms in labeled_df.columns) else ""
            print(f" - {nice:25s}: Accuracy = {acc:6.2f}% | Macro-F1 = {f1:6.2f}%{ms_txt}")
        print(f" - Rata-rata waktu PCD v2   : {labeled_df['PCD_ms'].mean():.2f} ms/citra")

    print(f"\nTabel dan visualisasi hasil tersimpan di: {DEMO_OUT_DIR}")

## 4. Mode 2: Live Demo Upload 1 Foto Langsung di Kelas
Jalankan sel ini saat presentasi jika dosen ingin mengetes langsung 1 foto daun baru dari laptop atau HP. Pilih file foto (`.jpg` atau `.png`), lalu dashboard 5 panel beserta konsensus prediksi langsung tampil otomatis.

In [ ]:
# === BAGIAN 4 (MODE 2): LIVE DEMO UPLOAD 1 FOTO LANGSUNG SAAT PRESENTASI ===
print("Pilih 1 foto daun baru (.jpg / .png) dari perangkat Anda:")
uploaded_live = files.upload()

for fname in uploaded_live.keys():
    img_live = cv2.imread(fname)
    if img_live is None:
        print(f"Gagal membaca file: {fname}")
        continue
    res_live = run_multimodel_inference(img_live, filename=fname, true_label=None)
    save_live = os.path.join(DEMO_OUT_DIR, f"LiveUpload_{os.path.splitext(fname)[0]}.png")
    visualize_demo_result(res_live, save_path=save_live)

    print(f"\n>>> HASIL LIVE DEMO ({fname}) <<<")
    print(f"PCD v2 (Gamma={res_live['pcd']['gamma_val']:.2f}, Area Daun={res_live['pcd']['leaf_area_pct']:.1f}%) selesai dalam {res_live['pcd']['pcd_ms']:.1f} ms")
    for m_key, val in res_live['preds'].items():
        print(f" - {m_key:12s}: {val['label']:10s} (Confidence: {val['conf']:5.1f}% | Latensi: {val['ms']:5.1f} ms)")
    print(f" => KONSENSUS FINAL: {res_live['consensus']} ({res_live['agreement']})")

## 5. Mode 3 (Opsional): Live Demo Capture Webcam Laptop di Google Colab
Gunakan sel ini apabila Anda membawa helai daun fisik dan kertas putih polos saat presentasi. Klik **Run**, arahkan kamera ke daun, tekan tombol **Ambil Foto Daun**, dan sistem langsung menjalankan pipeline PCD v2 beserta klasifikasi multi-model secara *real-time*.

In [ ]:
# === BAGIAN 5 (MODE 3): LIVE CAPTURE WEBCAM LAPTOP DI GOOGLE COLAB ===
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo_webcam(filename='/content/webcam_leaf.jpg', quality=0.92):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Ambil Foto Daun';
            capture.style.padding = '8px 18px';
            capture.style.fontSize = '15px';
            capture.style.fontWeight = 'bold';
            capture.style.marginBottom = '8px';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.maxWidth = '560px';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

try:
    cam_path = take_photo_webcam('/content/webcam_leaf.jpg')
    img_cam = cv2.imread(cam_path)
    res_cam = run_multimodel_inference(img_cam, filename='webcam_leaf.jpg', true_label=None)
    save_cam = os.path.join(DEMO_OUT_DIR, f"Webcam_{int(time.time())}.png")
    visualize_demo_result(res_cam, save_path=save_cam)
    print(f"\n>>> KONSENSUS WEBCAM: {res_cam['consensus']} ({res_cam['agreement']}) <<<")
except Exception as err:
    print("Gagal mengakses webcam (pastikan izin kamera browser aktif):", err)